<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/intermediate/04_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building AI Agents

Design and implement production-ready AI agents — from the ReAct loop to multi-step tool use with error recovery.

**Topics:** ReAct pattern, OpenAI function calling, custom tools, agent loops, error recovery, parallel tool execution

## 1. ReAct Pattern — Reason + Act Loop

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

REACT_SYSTEM = """You are a helpful assistant with access to tools.
Use this format to solve problems:

Thought: What do I need to figure out?
Action: tool_name({"param": "value"})
Observation: [tool result]
... (repeat as needed)
Thought: I now know the final answer.
Final Answer: [your answer]

Available tools: {tools}"""

MOCK_TOOLS = {
    'search': lambda q: f'Search results for "{q}": Found 3 relevant documents about {q}.',
    'calculator': lambda expr: str(eval(expr)),
    'get_weather': lambda city: f'{city}: 22°C, partly cloudy, 65% humidity.',
    'get_stock_price': lambda ticker: f'{ticker}: $142.35 (+1.2% today)',
}

# Simulated ReAct trace for a complex question
react_trace = """
User: What is the weather in Paris and what is 2^10?

Thought: The user is asking two separate things: weather in Paris and a calculation.
         I'll handle them one at a time.
Action: get_weather({"city": "Paris"})
Observation: Paris: 18°C, overcast, 70% humidity.

Thought: Got the weather. Now I need to calculate 2^10.
Action: calculator({"expr": "2**10"})
Observation: 1024

Thought: I have both answers now. I can provide the final response.
Final Answer: The weather in Paris is 18°C and overcast.
              2^10 = 1024.
"""

print('ReAct (Reason+Act) pattern:')
print(react_trace)
print('Key properties:')
print('  - Interleaved reasoning and action (no planning upfront)')
print('  - Each observation informs next thought')
print('  - Transparent: reasoning is visible for debugging')

## 2. OpenAI Function Calling / Tools API

In [ ]:
import json
from typing import Any, Callable

# Tool definitions (JSON Schema format)
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'search_documents',
            'description': 'Search the knowledge base for relevant documents.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Search query'},
                    'top_k': {'type': 'integer', 'description': 'Number of results', 'default': 3},
                    'filter_topic': {'type': 'string', 'description': 'Optional topic filter', 'enum': ['ml', 'nlp', 'cv', 'rl']},
                },
                'required': ['query'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'run_python_code',
            'description': 'Execute Python code and return the output.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'code': {'type': 'string', 'description': 'Python code to execute'},
                    'timeout': {'type': 'integer', 'description': 'Timeout in seconds', 'default': 30},
                },
                'required': ['code'],
            },
        },
    },
]

# Tool registry
TOOL_REGISTRY: dict[str, Callable] = {}

def register_tool(name: str):
    def decorator(fn: Callable) -> Callable:
        TOOL_REGISTRY[name] = fn
        return fn
    return decorator

@register_tool('search_documents')
def search_documents(query: str, top_k: int = 3, filter_topic: str = None) -> str:
    results = [f'Result {i}: {query} related content' for i in range(top_k)]
    return json.dumps({'results': results, 'query': query})

@register_tool('run_python_code')
def run_python_code(code: str, timeout: int = 30) -> str:
    import subprocess
    try:
        result = subprocess.run(['python', '-c', code], capture_output=True, text=True, timeout=timeout)
        return result.stdout or result.stderr
    except subprocess.TimeoutExpired:
        return f'Error: Code execution timed out after {timeout}s'

print('Tool registry:', list(TOOL_REGISTRY.keys()))
print()
print('How function calling works:')
print('  1. Send tools + messages to API')
print('  2. Model returns tool_calls (name + args) if it wants to use a tool')
print('  3. You execute the tool locally and send result back')
print('  4. Model incorporates observation and continues')

## 3. Agent Loop with Error Recovery

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class AgentStep:
    thought: str
    tool_name: Optional[str] = None
    tool_args: Optional[dict] = None
    observation: Optional[str] = None
    error: Optional[str] = None

class ToolAgent:
    """Production-grade agent with tool use, error recovery, and step limits."""
    
    def __init__(
        self,
        system_prompt: str,
        tools: list[dict],
        tool_registry: dict,
        model: str = 'gpt-4o-mini',
        max_steps: int = 10,
    ):
        self.system = system_prompt
        self.tools = tools
        self.registry = tool_registry
        self.model = model
        self.max_steps = max_steps
    
    def run(self, task: str) -> tuple[str, list[AgentStep]]:
        messages = [
            {'role': 'system', 'content': self.system},
            {'role': 'user', 'content': task},
        ]
        steps: list[AgentStep] = []
        
        for step_num in range(self.max_steps):
            response = client.chat.completions.create(
                model=self.model, messages=messages, tools=self.tools, tool_choice='auto',
            )
            msg = response.choices[0].message
            
            if msg.tool_calls is None:
                return msg.content, steps  # final answer
            
            messages.append({'role': 'assistant', 'content': msg.content, 'tool_calls': msg.tool_calls})
            
            for tool_call in msg.tool_calls:
                step = AgentStep(thought=msg.content or '', tool_name=tool_call.function.name)
                try:
                    args = json.loads(tool_call.function.arguments)
                    step.tool_args = args
                    result = self.registry[tool_call.function.name](**args)
                    step.observation = result
                except KeyError:
                    step.error = f'Unknown tool: {tool_call.function.name}'
                    result = step.error
                except Exception as e:
                    step.error = f'Tool error: {str(e)}'
                    result = step.error  # agent sees the error and can recover
                
                steps.append(step)
                messages.append({'role': 'tool', 'tool_call_id': tool_call.id, 'content': str(result)})
        
        return 'Max steps reached without final answer.', steps

print('Agent loop structure:')
print('  for step in range(max_steps):')
print('      response = llm(messages + tools)')
print('      if no tool_calls: return final answer')
print('      for each tool_call:')
print('          result = execute_tool(name, args)')
print('          append result to messages')
print()
print('Error recovery: agent receives the error message and can:')
print('  - Retry with different parameters')
print('  - Use a different tool')
print('  - Acknowledge failure in final answer')

## 4. Parallel Tool Execution

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

async def execute_tool_async(tool_call, registry: dict) -> tuple[str, str]:
    """Execute a single tool asynchronously."""
    try:
        args = json.loads(tool_call.function.arguments)
        tool_fn = registry[tool_call.function.name]
        
        # Run synchronous tools in thread pool to not block event loop
        result = await asyncio.get_event_loop().run_in_executor(None, lambda: tool_fn(**args))
        return tool_call.id, str(result)
    except Exception as e:
        return tool_call.id, f'Error: {e}'

async def agent_step_parallel(messages: list, tools: list, registry: dict) -> tuple:
    """Execute multiple tool calls in parallel (model may request several at once)."""
    response = await async_client.chat.completions.create(
        model='gpt-4o', messages=messages, tools=tools, tool_choice='auto',
    )
    msg = response.choices[0].message
    
    if not msg.tool_calls:
        return msg.content, []
    
    # Run all tool calls in parallel!
    results = await asyncio.gather(*[
        execute_tool_async(tc, registry) for tc in msg.tool_calls
    ])
    
    return None, results

# Latency comparison
tool_latencies = {'search': 200, 'get_weather': 150, 'get_stock_price': 100}
sequential_ms = sum(tool_latencies.values())  # 450ms
parallel_ms = max(tool_latencies.values())    # 200ms

print('Parallel tool execution:')
print(f'  Tools: {list(tool_latencies.keys())}')
print(f'  Sequential: {sequential_ms}ms')
print(f'  Parallel:   {parallel_ms}ms')
print(f'  Speedup:    {sequential_ms/parallel_ms:.1f}x')
print()
print('When gpt-4o makes multiple tool_calls in one response:')
print('  → Execute them all with asyncio.gather() for max throughput')
print('  → Return all results before next LLM call')

## 5. Building a Multi-Step Research Agent

In [ ]:
from typing import Callable

# Research agent tool suite
RESEARCH_TOOLS = [
    {'type': 'function', 'function': {
        'name': 'web_search', 'description': 'Search the web for current information.',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']},
    }},
    {'type': 'function', 'function': {
        'name': 'read_url', 'description': 'Fetch and read the content of a URL.',
        'parameters': {'type': 'object', 'properties': {'url': {'type': 'string'}}, 'required': ['url']},
    }},
    {'type': 'function', 'function': {
        'name': 'summarize_text', 'description': 'Summarize a long text into key points.',
        'parameters': {'type': 'object', 'properties': {'text': {'type': 'string'}, 'max_words': {'type': 'integer', 'default': 100}}, 'required': ['text']},
    }},
    {'type': 'function', 'function': {
        'name': 'save_to_report', 'description': 'Save a finding to the research report.',
        'parameters': {'type': 'object', 'properties': {'section': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['section', 'content']},
    }},
]

RESEARCH_SYSTEM = """You are an expert research assistant. Your task: research the given topic thoroughly.

Your process:
1. Search for multiple angles of the topic
2. Read the most relevant sources
3. Summarize key findings
4. Save findings to the report using save_to_report()
5. Provide a final comprehensive summary

Be thorough — use 5-10 tool calls before writing the final answer."""

# Simulated research trace
research_trace = [
    {'step': 1, 'tool': 'web_search', 'args': {'query': 'transformer architecture 2023 advances'}, 'observation': 'Found 10 results about Flash Attention, RoPE, GQA...'},
    {'step': 2, 'tool': 'web_search', 'args': {'query': 'LLM inference optimization techniques'}, 'observation': 'Speculative decoding, continuous batching, PagedAttention...'},
    {'step': 3, 'tool': 'read_url', 'args': {'url': 'https://arxiv.org/abs/2307.09288'}, 'observation': 'LLaMA 2 paper: 70B model achieves GPT-3.5 performance...'},
    {'step': 4, 'tool': 'save_to_report', 'args': {'section': 'Architecture Advances', 'content': 'Flash Attention 2 reduces memory 4x...'}, 'observation': 'Saved.'},
    {'step': 5, 'tool': 'save_to_report', 'args': {'section': 'Inference Optimization', 'content': 'PagedAttention (vLLM) achieves 24x throughput...'}, 'observation': 'Saved.'},
]

print('Research agent trace (5 of 8 steps):')
for step in research_trace:
    args_str = json.dumps(step['args'])[:60]
    print(f'  Step {step["step"]}: {step["tool"]}({args_str}...)')
    print(f'           → {step["observation"][:70]}...')
print()
print('Agent autonomously plans tool sequence based on what it learns.')

## Exercises

1. **Code execution agent:** Build an agent with a Python REPL tool (using `subprocess` in a Docker container for safety). Have it solve math problems by writing and executing Python code, handling syntax errors and runtime exceptions gracefully.

2. **Agent evaluation:** Create a benchmark of 20 tasks with known correct answers. Run your agent on all 20 tasks and measure: success rate, average number of steps, average latency, and cost per task.

3. **Tool caching:** Add a caching layer to the agent that caches tool results keyed by `(tool_name, sorted(args.items()))`. Measure cache hit rate and latency improvement on repeated queries.